In [1]:
import requests
import pandas as pd

URL = "http://andromeda.lasdpc.icmc.usp.br:61102/questdb"

def get_from_source(sql):
    response = requests.get(
        f"{URL}/exec",
        params={"query": sql},
        timeout=30
    )

    print("Status:", response.status_code)
    print("URL:", response.url)
    print("Resposta:", response.text[:500])

    response.raise_for_status()

    data = response.json()

    columns = [col["name"] for col in data["columns"]]
    return pd.DataFrame(data["dataset"], columns=columns)

# Descobrir as tabelas reais
db = get_from_source("SHOW TABLES")

display(db)

Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=SHOW+TABLES
Resposta: {"query":"SHOW TABLES","columns":[{"name":"table_name","type":"STRING"}],"timestamp":-1,"dataset":[["sensor_humidity"],["sensor_temperature"],["sensor_energy"]],"count":3}


,table_name
0,sensor_humidity
1,sensor_temperature
2,sensor_energy


In [2]:
tabelas = [
    "sensor_temperature",
    "sensor_humidity",
    "sensor_energy"
]

for tabela in tabelas:
    print(f"\n=== {tabela} ===")

    sensores = get_from_source(f"""
        SELECT
            environment,
            sensor_id,
            sensor_model,
            count(*) AS total
        FROM {tabela}
        GROUP BY environment, sensor_id, sensor_model
        ORDER BY environment, sensor_id
    """)

    display(sensores)


=== sensor_temperature ===
Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=%0A++++++++SELECT%0A++++++++++++environment%2C%0A++++++++++++sensor_id%2C%0A++++++++++++sensor_model%2C%0A++++++++++++count%28%2A%29+AS+total%0A++++++++FROM+sensor_temperature%0A++++++++GROUP+BY+environment%2C+sensor_id%2C+sensor_model%0A++++++++ORDER+BY+environment%2C+sensor_id%0A++++
Resposta: {"query":"\n        SELECT\n            environment,\n            sensor_id,\n            sensor_model,\n            count(*) AS total\n        FROM sensor_temperature\n        GROUP BY environment, sensor_id, sensor_model\n        ORDER BY environment, sensor_id\n    ","columns":[{"name":"environment","type":"SYMBOL"},{"name":"sensor_id","type":"SYMBOL"},{"name":"sensor_model","type":"SYMBOL"},{"name":"total","type":"LONG"}],"timestamp":-1,"dataset":[["1006","1","DHT22",594465],["1006","15","DH


,environment,sensor_id,sensor_model,total
0,1006,1,DHT22,594465
1,1006,15,DHT22,123573
2,1006,48,DHT22,508066
3,1006,86782,DHT22,570487
4,1006,86789,DHT22,495946
5,1006,86790,DHT22,516968
6,1006,86791,DHT22,518720
7,1006,86793,DHT22,517005
8,1006,887,DHT22,518120
9,1006,891,DHT22,510688



=== sensor_humidity ===
Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=%0A++++++++SELECT%0A++++++++++++environment%2C%0A++++++++++++sensor_id%2C%0A++++++++++++sensor_model%2C%0A++++++++++++count%28%2A%29+AS+total%0A++++++++FROM+sensor_humidity%0A++++++++GROUP+BY+environment%2C+sensor_id%2C+sensor_model%0A++++++++ORDER+BY+environment%2C+sensor_id%0A++++
Resposta: {"query":"\n        SELECT\n            environment,\n            sensor_id,\n            sensor_model,\n            count(*) AS total\n        FROM sensor_humidity\n        GROUP BY environment, sensor_id, sensor_model\n        ORDER BY environment, sensor_id\n    ","columns":[{"name":"environment","type":"SYMBOL"},{"name":"sensor_id","type":"SYMBOL"},{"name":"sensor_model","type":"SYMBOL"},{"name":"total","type":"LONG"}],"timestamp":-1,"dataset":[["1006","1","DHT22",635937],["1006","15","DHT22


,environment,sensor_id,sensor_model,total
0,1006,1,DHT22,635937
1,1006,15,DHT22,129919
2,1006,48,DHT22,737100
3,1006,86782,DHT22,611036
4,1006,86789,DHT22,728169
5,1006,86790,DHT22,762446
6,1006,86791,DHT22,764156
7,1006,86793,DHT22,762060
8,1006,887,DHT22,762361
9,1006,891,DHT22,740762



=== sensor_energy ===
Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=%0A++++++++SELECT%0A++++++++++++environment%2C%0A++++++++++++sensor_id%2C%0A++++++++++++sensor_model%2C%0A++++++++++++count%28%2A%29+AS+total%0A++++++++FROM+sensor_energy%0A++++++++GROUP+BY+environment%2C+sensor_id%2C+sensor_model%0A++++++++ORDER+BY+environment%2C+sensor_id%0A++++
Resposta: {"query":"\n        SELECT\n            environment,\n            sensor_id,\n            sensor_model,\n            count(*) AS total\n        FROM sensor_energy\n        GROUP BY environment, sensor_id, sensor_model\n        ORDER BY environment, sensor_id\n    ","columns":[{"name":"environment","type":"SYMBOL"},{"name":"sensor_id","type":"SYMBOL"},{"name":"sensor_model","type":"SYMBOL"},{"name":"total","type":"LONG"}],"timestamp":-1,"dataset":[["1006","2","ACS712",337009],["1006","23","ACS712


,environment,sensor_id,sensor_model,total
0,1006,2,ACS712,337009
1,1006,23,ACS712,415438


In [ ]:
import json
import pandas as pd

# Carregar informações cadastrais
with open("sensores.json", "r", encoding="utf-8") as f:
    dados_sensores = json.load(f)

cadastro = pd.DataFrame(dados_sensores)

# Renomear campos para facilitar o cruzamento
cadastro = cadastro.rename(columns={
    "id": "sensor_id",
    "name": "nome",
    "description": "descricao",
    "id_enviroment": "ambiente_cadastro",
    "available": "disponivel"
})

# Buscar sensores com registros no QuestDB
tabelas = [
    "sensor_temperature",
    "sensor_humidity",
    "sensor_energy"
]

resultados = []

for tabela in tabelas:
    df = get_from_source(f"""
        SELECT
            environment,
            sensor_id,
            sensor_model,
            count(*) AS total
        FROM {tabela}
        GROUP BY environment, sensor_id, sensor_model
    """)

    df["tipo_medicao"] = tabela.replace("sensor_", "")
    resultados.append(df)

# Consolidar todas as tabelas
registros = pd.concat(resultados, ignore_index=True)

registros["sensor_id"] = pd.to_numeric(
    registros["sensor_id"], errors="coerce"
).astype("Int64")

cadastro["sensor_id"] = pd.to_numeric(
    cadastro["sensor_id"], errors="coerce"
).astype("Int64")

# Cruzar com o cadastro, mantendo TODOS os sensores com registros
sensores_completos = registros.merge(
    cadastro,
    on="sensor_id",
    how="left",
    validate="many_to_one"
)

# Organizar colunas
colunas = [
    "environment",
    "ambiente_cadastro",
    "sensor_id",
    "nome",
    "descricao",
    "sensor_model",
    "tipo_medicao",
    "disponivel",
    "total"
]

sensores_completos = sensores_completos[colunas]

display(
    sensores_completos.sort_values(
        ["environment", "nome", "tipo_medicao"]
    )
)

Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=%0A++++++++SELECT%0A++++++++++++environment%2C%0A++++++++++++sensor_id%2C%0A++++++++++++sensor_model%2C%0A++++++++++++count%28%2A%29+AS+total%0A++++++++FROM+sensor_temperature%0A++++++++GROUP+BY+environment%2C+sensor_id%2C+sensor_model%0A++++
Resposta: {"query":"\n        SELECT\n            environment,\n            sensor_id,\n            sensor_model,\n            count(*) AS total\n        FROM sensor_temperature\n        GROUP BY environment, sensor_id, sensor_model\n    ","columns":[{"name":"environment","type":"SYMBOL"},{"name":"sensor_id","type":"SYMBOL"},{"name":"sensor_model","type":"SYMBOL"},{"name":"total","type":"LONG"}],"timestamp":-1,"dataset":[["1006","891","DHT22",510688],["1006","86782","DHT22",570487],["1006","48","DHT22",50
Status: 200
URL: http://andromeda.lasdpc.icmc.usp.br:61102/questdb/exec?query=%0A++++++++SELECT%0A++++++++++++environment%2C%0A++++++++++++sensor_id%2C%0A++++++++++++se

ValueError: You are trying to merge on str and int64 columns for key 'sensor_id'. If you wish to proceed you should use pd.concat